<a href="https://colab.research.google.com/github/Mahnoor-Kalsoom/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahnoor-Kalsoom/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

My baseline rule identifies content that may require optimization based on search visibility and user engagement.

The rule prioritizes content with:
- High Google Search Console (GSC) impressions,
- Low click-through behaviour,
- Poor average search position, or
- Low user engagement.

The purpose is to rank content items that are likely to benefit from manual review before applying machine learning models.

## Reason Codes

The rule can produce the following reason codes:

- LOW_CTR_HIGH_IMPRESSIONS
- LOW_VISIBILITY
- LOW_ENGAGEMENT
- REVIEW_PRIORITY

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*



In [5]:
!pip install duckdb pyarrow -q

import duckdb
from google.colab import userdata

# Get your Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Create Hugging Face secret
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

# Dataset location
REL = "hf://datasets/FlyRank/internship-warehouse"

print("Connection successful!")

Connection successful!


In [6]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,78835655


In [7]:
import os
import pandas as pd

query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    scroll_events,
    sessions_ai
FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2025-01'
AND gsc_data_available IS TRUE
"""

df = con.sql(query).df()

# Fill missing values
df = df.fillna(0)

# ------------------------
# Baseline Score
# ------------------------

df["baseline_score"] = (
      df["gsc_impressions"] * 0.4
    + df["gsc_clicks"] * 0.2
    + df["scroll_events"] * 0.2
    - df["gsc_sum_position"] * 0.2
)

# ------------------------
# Reason Code
# ------------------------

df["reason_code"] = "REVIEW_PRIORITY"

df.loc[
    (df["gsc_impressions"] > 100)
    & (df["gsc_clicks"] < 10),
    "reason_code"
] = "LOW_CTR_HIGH_IMPRESSIONS"

df.loc[
    df["scroll_events"] < 5,
    "reason_code"
] = "LOW_ENGAGEMENT"

# ------------------------
# Action Label
# ------------------------

df["action"] = "Review"

df.loc[
    df["reason_code"]=="LOW_CTR_HIGH_IMPRESSIONS",
    "action"
] = "Improve CTR"

df.loc[
    df["reason_code"]=="LOW_ENGAGEMENT",
    "action"
] = "Improve Engagement"

# ------------------------
# Ranking
# ------------------------

df = df.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(df.head())

     report_date           client_hash_id           content_hash_id  \
1221  2025-01-31  client_9958f0a7ae1df715  content_f94fe855380e150f   
1019  2025-01-30  client_9958f0a7ae1df715  content_f94fe855380e150f   
446   2025-01-28  client_9958f0a7ae1df715  content_f94fe855380e150f   
146   2025-01-27  client_9958f0a7ae1df715  content_f94fe855380e150f   
766   2025-01-29  client_9958f0a7ae1df715  content_f94fe855380e150f   

      gsc_impressions  gsc_clicks  gsc_sum_position  scroll_events  \
1221              305           5               484              0   
1019              247           6               409              0   
446               303           3               549              0   
146               266           6               482              0   
766               240           5               434              0   

      sessions_ai  baseline_score     reason_code              action  
1221            0            26.2  LOW_ENGAGEMENT  Improve Engagement  
1019    

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The table below reviews the twenty highest ranked content items produced by the baseline rule.

Each row includes:

- Recommended action
- Reason code
- Confidence note
- What could make the recommendation incorrect

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = df.head(20).copy()

top20["confidence"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Seasonal traffic, incomplete data, temporary ranking fluctuations or missing engagement information."
)

top20[
[
"content_hash_id",
"action",
"reason_code",
"confidence",
"what_would_make_it_wrong"
]
]


,content_hash_id,action,reason_code,confidence,what_would_make_it_wrong
1221,content_f94fe855380e150f,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
1019,content_f94fe855380e150f,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
446,content_f94fe855380e150f,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
146,content_f94fe855380e150f,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
766,content_f94fe855380e150f,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
80,content_84f5a9ecfefa108e,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
134,content_1b1f69effd0e4531,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
709,content_f3a75d8cf58dd50b,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
1232,content_17cb95e09b288a79,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."
392,content_f3a75d8cf58dd50b,Improve Engagement,LOW_ENGAGEMENT,Medium,"Seasonal traffic, incomplete data, temporary r..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some highly ranked pages may not actually require action.

Examples include:

- Seasonal search behaviour
- Temporary ranking changes
- Incomplete analytics collection
- Recently published content with limited history

These situations may produce high baseline scores without representing genuine optimization opportunities.

## Leakage Check

The baseline rule uses only historical observations available at decision time.

No future windows, future labels, or product-generated flags were used when computing the score.

Therefore, no information leakage was intentionally introduced into the baseline ranking.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Rows ranked:", len(df))

print()

print("Reason Code Distribution")

display(
df["reason_code"].value_counts()
)

print()

print("CSV saved to")

print("work/outputs/baseline_action_score.csv")


Rows ranked: 1297

Reason Code Distribution


,count
reason_code,
LOW_ENGAGEMENT,1297



CSV saved to
work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.